# 01 Environment Setup and Imports

In [1]:
%pip install groq python-dotenv langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq

## LLMs Setup

In [3]:
load_dotenv(find_dotenv(usecwd=True))

reasoning_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_REASONING"),
    temperature=0.5,
    reasoning_effort="high",
)

responding_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY_RESPONDING"),
    temperature=0.3,
)

# Data prep: a structured "plan" task, so medium reasoning is enough
prep_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_PREPARING"),
    temperature=0.2,
    reasoning_effort="medium",
)

consultant_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_CONSULTING"),
    temperature=0.3,
    reasoning_effort="medium",
)

In [4]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

for folder in (ROOT / "src").iterdir():
    if folder.is_dir():
        sys.path.append(str(folder))

from loader import load_file
from data_prep import plan_data_prep, apply_prep_plan, prep_to_text
from analysis import (
    descriptive_analysis,
    group_analysis,
    ratio_analysis,
    time_series_analysis,
    correlation_analysis,
    detect_anomalies,
)
from consultant import consult, format_report
from powerbi_export import RunRecord, export_runs

print("ROOT:", ROOT)
print("Imports OK ✅")

ROOT: c:\Users\Viola Ibrahim\Analytix AI\analytix-ai
Imports OK ✅


In [5]:
df_raw = load_file(str(ROOT / "data" / "ecommerce_sales_customer_analytics_150k.csv"))

prep_plan = plan_data_prep(df_raw, prep_llm)
df, prep_log = apply_prep_plan(df_raw, prep_plan)

print("Summary:", prep_plan.dataset_summary)
print("Shape before/after:", df_raw.shape, "/", df.shape)
print("Duplicates removed:", prep_log["duplicates_removed"])
print("Empty columns dropped:", prep_log["empty_columns_dropped"])
print("Dates converted:", prep_log["dates_converted"], "| refused:", prep_log["dates_refused"])

print("\nROLES:")
for c in prep_plan.columns:
    print(f"  {c.column:28s} {c.role:11s} {c.reason}")

print("\nNULL HANDLING:")
for a in prep_log["actions"]:
    print("  ", a)

print("\nREFUSED by code guards:")
for r in prep_log["refused"]:
    print("  ", r)

print("\nRELATIONSHIPS:")
for r in prep_plan.relationships:
    print("  ", r.columns, "->", r.description)

[data prep] attempt 1/4 failed (no structured output), retrying...
[data prep] rate limit, waiting 10s (attempt 2/4)...
[data prep] rate limit, waiting 10s (attempt 3/4)...
[data prep] rate limit, waiting 10s (attempt 4/4)...
[data prep] LLM plan unavailable, using safe defaults for every column.
Summary: (fallback: no LLM plan)
Shape before/after: (138116, 46) / (138116, 46)
Duplicates removed: 0
Empty columns dropped: []
Dates converted: [] | refused: []

ROLES:

NULL HANDLING:
   delivery_days: 17.8% nulls -> keep_nan
   estimated_delivery_days: 17.8% nulls -> keep_nan
   return_status: 93.1% nulls -> fill_label 'Missing'
   return_reason: 93.1% nulls -> fill_label 'Missing'
   customer_rating: 17.8% nulls -> keep_nan
   review_sentiment: 17.8% nulls -> fill_label 'Missing'
   customer_review: 17.8% nulls -> fill_label 'Missing'
   campaign_name: 60.3% nulls -> fill_label 'Missing'
   coupon_code: 80.0% nulls -> fill_label 'Missing'

REFUSED by code guards:
   delivery_days: no plan

In [6]:
profile_text = prep_to_text(prep_plan, df)
print(profile_text)

Summary: (fallback: no LLM plan)
Rows: 138116, Columns: 46
Date columns: []
Metric columns (numbers to sum/average): []
Dimension columns (categories to group by): []
Identifier columns (never analyze): []
Other columns: ['order_id', 'order_date', 'order_time', 'order_status', 'sales_channel', 'customer_id', 'customer_name', 'customer_age', 'gender', 'customer_segment', 'customer_type', 'customer_city', 'customer_state', 'customer_country', 'region', 'customer_postal_code', 'payment_method', 'payment_status', 'currency', 'shipping_method', 'warehouse', 'delivery_days', 'estimated_delivery_days', 'delivery_status', 'return_status', 'return_reason', 'customer_rating', 'review_sentiment', 'customer_review', 'marketing_channel', 'campaign_name', 'coupon_code', 'loyalty_points_earned', 'loyalty_points_redeemed', 'quantity', 'gross_sales', 'discount_amount', 'tax_amount', 'shipping_cost', 'net_sales', 'product_cost', 'profit', 'profit_margin_percentage', 'customer_lifetime_value', 'is_repeat

In [7]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# ====== 1) Tools the planner is allowed to use ======
TOOLS_DESCRIPTION = """
- descriptive_analysis(column): summary statistics for ONE numeric or categorical column
- group_analysis(metric, group_by, agg): aggregate a numeric column (metric) by a categorical column (group_by); agg is one of sum/mean/count/min/max
- ratio_analysis(numerator, denominator, group_by): ratio of the SUMS of two numeric columns per group. Use it for margins, rates and efficiency, e.g. profit margin = profit / net_sales per region. Prefer it over averaging a percentage column.
- time_series_analysis(date_column, metric, freq): aggregate a numeric column over time (needs a date column)
- correlation_analysis(): correlation matrix for all numeric columns
- detect_anomalies(column, threshold): find outliers in ONE numeric column
"""

# ====== 2) Shape of the plan the model must return ======
class PlanStep(BaseModel):
    step_id: int = Field(description="Step number starting from 1")
    tool: Literal[
        "descriptive_analysis",
        "group_analysis",
        "ratio_analysis",
        "time_series_analysis",
        "correlation_analysis",
        "detect_anomalies",
    ] = Field(description="Name of the tool to run")
    column: Optional[str] = Field(default=None, description="Column name (descriptive_analysis, detect_anomalies)")
    metric: Optional[str] = Field(default=None, description="Numeric column to aggregate (group_analysis, time_series_analysis)")
    group_by: Optional[str] = Field(default=None, description="Categorical column to group by (group_analysis, ratio_analysis)")
    agg: Optional[str] = Field(default=None, description="sum, mean, count, min or max (group_analysis)")
    numerator: Optional[str] = Field(default=None, description="Numeric column on top of the ratio (ratio_analysis)")
    denominator: Optional[str] = Field(default=None, description="Numeric column at the bottom of the ratio (ratio_analysis)")
    date_column: Optional[str] = Field(default=None, description="Date column (time_series_analysis)")
    goal: str = Field(description="What this step answers, in one short sentence")

class AnalysisPlan(BaseModel):
    can_answer: bool = Field(description="False if the dataset cannot answer the question")
    reason_if_not: Optional[str] = Field(default=None, description="Why it cannot be answered")
    steps: List[PlanStep] = Field(default_factory=list)

# ====== 3) Planner prompt ======
planner_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a senior data analyst. You only PLAN, you never execute.\n"
     "Write the shortest ordered plan that answers the user's question using ONLY these tools:\n"
     "{tools}\n\n"
     "Rules:\n"
     "- Use ONLY column names that appear in the dataset description. Never invent columns.\n"
     "- Never analyze the columns listed as identifier columns.\n"
     "- When the question or the business goal involves profitability, efficiency or return on spend, "
     "include a ratio_analysis step (for example profit / net_sales) next to the totals, "
     "so groups are compared on efficiency and not only on size.\n"
     "- If the dataset cannot answer the question, set can_answer=false and explain why.\n"
     "- Write every text field in English only.\n\n"
     "Dataset description:\n{profile}"),
    ("human", "{question}"),
])

# ====== 4) Planner chain (lighter model: planning is simple) ======
planner_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_REASONING"),
    temperature=0.3,
    reasoning_effort="medium",
)
planner_chain = planner_prompt | planner_llm.with_structured_output(AnalysisPlan)

def plan_analysis(question: str) -> AnalysisPlan:
    return planner_chain.invoke({
        "tools": TOOLS_DESCRIPTION,
        "profile": profile_text,
        "question": question,
    })

In [8]:
# ====== Executor: runs the plan using your real Phase 1 functions ======

def run_step(step: PlanStep, df):
    """Run ONE plan step and return its result."""
    if step.tool == "descriptive_analysis":
        return descriptive_analysis(df, step.column)

    if step.tool == "group_analysis":
        return group_analysis(df, step.metric, step.group_by, step.agg or "sum")

    if step.tool == "ratio_analysis":
        return ratio_analysis(df, step.numerator, step.denominator, step.group_by)

    if step.tool == "time_series_analysis":
        return time_series_analysis(df, step.date_column, step.metric, freq="ME")

    if step.tool == "correlation_analysis":
        return correlation_analysis(df)

    if step.tool == "detect_anomalies":
        return detect_anomalies(df, step.column)

    raise ValueError(f"Unknown tool: {step.tool}")


def execute_plan(plan: AnalysisPlan, df) -> list:
    """Run all steps in order. A failed step is recorded instead of crashing everything."""
    results = []

    if not plan.can_answer:
        print("Cannot answer:", plan.reason_if_not)
        return results

    for step in plan.steps:
        try:
            output = run_step(step, df)
            results.append({"step": step, "output": output, "error": None})
        except Exception as e:
            results.append({"step": step, "output": None, "error": str(e)})

    return results


In [9]:
from langchain_core.prompts import ChatPromptTemplate

# ====== 2) Insight prompt: explain ONLY what the numbers show ======
insight_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data analyst explaining results to a non-technical user.\n"
     "Rules:\n"
     "- Use ONLY the numbers in the analysis results below. Never invent numbers.\n"
     "- Answer the user's question directly first, then add 2-3 short key insights.\n"
     "- If a step has an ERROR, mention it briefly instead of guessing.\n"
     "- Keep it short and clear.\n"
     "- Answer in English only.\n\n"
     "Analysis results:\n{results}"),
    ("human", "{question}"),
])

# ====== 3) Insight chain: prompt -> responding model ======
insight_chain = insight_prompt | responding_llm


def generate_insights(question: str, results: list) -> str:
    response = insight_chain.invoke({
        "results": results_to_text(results),
        "question": question,
    })
    return response.content


# ====== 4) Full pipeline: question -> plan -> execute -> insights ======
def ask(question: str) -> str:
    plan = plan_analysis(question)
    results = execute_plan(plan, df)

    # If the planner said the question can't be answered, return its reason
    if not plan.can_answer:
        return f"I can't answer this with the current dataset: {plan.reason_if_not}"

    return generate_insights(question, results)


In [10]:
import numpy as np
import pandas as pd


# ====== Summarize each tool output in code before sending it to the LLM ======
def summarize_output(step, output, data) -> str:
    """Turn a raw tool output into a short, unambiguous text summary."""

    # Correlation: send only the top 10 strongest pairs, not the full matrix
    if step.tool == "correlation_analysis":
        id_cols = [c for c in output.columns if str(c).lower().endswith("id")]
        corr = output.drop(index=id_cols, columns=id_cols, errors="ignore")
        mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)  # upper triangle only
        pairs = corr.where(mask).stack()
        pairs = pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(10)
        lines = [f"{a} <-> {b}: {v:.3f}" for (a, b), v in pairs.items()]
        return "Top 10 strongest correlations (by absolute value):\n" + "\n".join(lines)

    # Anomalies: the returned rows ARE the outliers, so say it explicitly
    if step.tool == "detect_anomalies":
        col = step.column
        n_flagged = len(output)
        if n_flagged == 0:
            return f"No anomalies were detected in column '{col}'."
        return (
            f"{n_flagged} anomalous rows (outliers) were detected in '{col}' "
            f"out of {len(data)} total rows.\n"
            f"The outlier values range from {output[col].min()} to {output[col].max()}.\n"
            f"For comparison, the whole column ranges from {data[col].min()} to {data[col].max()} "
            f"with a median of {data[col].median()}."
        )

    # Group / ratio analysis: let the CODE decide if the gap between groups is meaningful
    if step.tool in ("group_analysis", "ratio_analysis") and isinstance(output, pd.Series) and len(output) > 1:
        text = output.to_string()
        lo, hi = float(output.min()), float(output.max())
        if lo > 0:
            gap = (hi - lo) / lo * 100
            verdict = (
                "NEGLIGIBLE (under 5%): treat the groups as practically equal and do NOT rank them."
                if gap < 5 else "meaningful."
            )
            text += (
                f"\nGap between the highest group ({output.idxmax()}) and the lowest group "
                f"({output.idxmin()}): {gap:.1f}% -> {verdict}"
            )
        return text

    # Everything else: Series / DataFrame / dict -> text
    return output.to_string() if hasattr(output, "to_string") else str(output)

def results_to_text(results: list) -> str:
    """Build the text block that is sent to the insight model."""
    parts = []
    for r in results:
        step = r["step"]
        parts.append(f"Step {step.step_id} ({step.tool}) - Goal: {step.goal}")
        if r["error"]:
            parts.append(f"ERROR: {r['error']}")
        else:
            parts.append(summarize_output(step, r["output"], df))
        parts.append("")
    return "\n".join(parts)


# ====== 3) Guard against empty model responses ======
def generate_insights(question: str, results: list) -> str:
    response = insight_chain.invoke({
        "results": results_to_text(results),
        "question": question,
    })
    if not response.content or not response.content.strip():
        return "(The model returned an empty response. Try running the question again.)"
    return response.content


## Consultant Agent

The agent itself lives in `src/agents/consultant.py`. The notebook only wires it into the pipeline.

In [11]:
# ====== Full pipeline: question + goal -> plan -> execute -> consult ======
if "RUNS" not in globals():   
    RUNS = []

def ask_consultant(question: str, goal: str):
    plan = plan_analysis(f"{question}\n(Business goal: {goal})")

    if not plan.can_answer:
        print(f"I can't answer this with the current dataset: {plan.reason_if_not}")
        return None

    results = execute_plan(plan, df)

    failed = [r for r in results if r["error"]]
    ok_results = [r for r in results if not r["error"]]

    if failed:
        print("Failed steps:")
        for r in failed:
            print(f"  step {r['step'].step_id} ({r['step'].tool}): {r['error']}")
        print()

    if not ok_results:
        print("No analysis step succeeded, so no business report was generated.")
        return None

    results_text = results_to_text(ok_results)

    try:
        report, verified = consult(question, goal, results_text, consultant_llm)
    except Exception as e:  # BadRequestError / RateLimitError after all retries
        print(f"Structured report failed ({type(e).__name__}), falling back to plain-text insights:\n")
        print(generate_insights(question, ok_results))
        return None

    RUNS.append(RunRecord(question=question, goal=goal, results=ok_results,
                      results_text=results_text, report=report, verified=verified))

    print(format_report(report, verified))
    return report, verified

In [12]:
_ = ask_consultant(
    question="Which regions generate the most net_sales and profit, and what is each region's profit margin?",
    goal="Increase overall profitability",
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m2qzabaaeynsgcsksb2jbhs5` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199874, Requested 1764. Please try again in 11m47.616s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Test

In [ ]:
_ = ask_consultant(
    question="Which product categories generate the most sales, and how does their profit margin compare?",
    goal="Increase overall profitability",
)


I can't answer this with the current dataset: The dataset does not contain a product category column, so we cannot determine which product categories generate the most sales or compare their profit margins.


In [ ]:
_ = ask_consultant(
    question="Which regions generate the most net_sales and the most profit?",
    goal="Increase overall profitability",
)

print("=" * 70)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01m2qzabaaeynsgcsksb2jbhs5` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199800, Requested 2618. Please try again in 17m24.576s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
_ = ask_consultant(
    question="Which marketing channels bring the highest net_sales, and how do their average profit compare?",
    goal="Decide where to invest the marketing budget",
)

print("=" * 70)

In [ ]:
_ = ask_consultant(
    question="How did net_sales change over time?",
    goal="Understand sales trends and plan ahead",
)


SITUATION: Net sales exhibit strong seasonal peaks in November and December each year, while remaining relatively stable and lower during the rest of the year, with modest upward movement in holiday peaks over the five‑year period.

KEY FINDINGS:
 - Net sales peak sharply in November and December each year, reaching over 4.2M (e.g., 2021-11: 4234660.71, 2021-12: 4900970.07) and staying above 4.3M in later years (e.g., 2025-11: 4270161.00, 2025-12: 4555033.50).
 - From January to October, net sales remain between roughly 1.9M and 3.2M, with the lowest points in February (e.g., 2021-02: 1905413.15, 2025-02: 2018064.34) and higher values in August (e.g., 2021-08: 3220659.48, 2025-08: 2958759.35).
 - Across the five-year period, the holiday-season peaks show a modest upward movement, from 4.23M in November 2021 to 4.27M in November 2025 and from 4.90M in December 2021 to 4.55M in December 2025, indicating sustained high demand during year‑end.
 - Outside the holiday months, net sales are r

In [ ]:
metric_cols = [c.column for c in prep_plan.columns if c.role == "metric"]
export_runs(RUNS, ROOT / "outputs" / "powerbi", df=df, metric_columns=metric_cols)

runs.csv: 4 rows
findings.csv: 12 rows
recommendations.csv: 10 rows
analysis_results.csv: 120 rows
kpis.csv: 17 rows


{'runs': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/runs.csv'),
 'findings': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/findings.csv'),
 'recommendations': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/recommendations.csv'),
 'analysis_results': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/analysis_results.csv'),
 'kpis': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/kpis.csv')}

In [ ]:
from dashboard import build_dashboard

metric_cols = [c.column for c in prep_plan.columns if c.role == "metric"]
build_dashboard(RUNS, ROOT / "outputs" / "dashboard.html", df=df, metric_columns=metric_cols)

In [ ]:
from evaluation import run_evaluation, load_cases

sensitive = [c.column for c in prep_plan.columns if c.role in ("sensitive", "free_text")]

results = run_evaluation(
    df_columns=df.columns,
    plan_fn=plan_analysis,
    execute_fn=lambda plan: execute_plan(plan, df),
    results_to_text_fn=results_to_text,
    consult_fn=lambda q, g, t: consult(q, g, t, consultant_llm),
    cases=load_cases(ROOT / "tests" / "eval_cases_ecommerce.json"),
    sensitive_columns=sensitive,
)

[1/8] regions_margin ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...
[2/8] sales_channel_margin ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...
[3/8] marketing_negligible ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...
[4/8] trend_over_time ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...
[5/8] delivery_vs_rating ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...
[6/8] missing_product_category ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...
[7/8] missing_churn ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...
[8/8] sensitive_gender ...
  [eval] rate limit, waiting 10s...
  [eval] rate limit, waiting 10s...

                      id  passed status tools_used  recs_unverified  findings_unverified  seconds
          regions_margin   False  error                           0                    0     25.8